In [1]:
#%pip install openai openrouter tiktoken tqdm pandas

# 2. Import the libraries
import openai
import openrouter
import tiktoken
import tqdm
import pandas as pd

In [2]:
#%pip install tiktoken openpyxl
import openpyxl

In [3]:
df_dirty_names = pd.read_excel(r"ENTER PATH NAME HERE", sheet_name="Dirty Data")

In [4]:
df_dirty_names.head()

,Part Name
0,#MM-18389 Wiper Blade SKU65302
1,1434 Engine Air Filter
2,NorthStar cartridge oil filter - universal fit
3,1711 CABIN AIR FILTER
4,2139 HD Alt.


In [5]:
df_map_names = pd.read_excel(r"ENTER PATH NAME HERE", sheet_name="Mapped Names")

In [6]:
df_map_names.head()

,Generic Name
0,Air Filter
1,Motor Oil
2,Oil Filter
3,Brake Pad
4,Spark Plug


In [8]:
dirty_names = sorted(set(df_dirty_names['Part Name'].dropna().tolist()))

In [9]:
len(dirty_names)

1000

In [11]:
map_names = sorted(set(df_map_names['Generic Name'].dropna().tolist()))

In [12]:
len(map_names)

10

In [13]:
map_names[:5]

['Air Filter', 'Alternator', 'Brake Pad', 'Car Battery', 'Headlight Bulb']

In [14]:
dirty_names[:5]

['  #BR-96537 WINDSHIELD WIPER BLADE',
 '  #MK-13224 HD spark-plug',
 '  #MM-18389 Wiper Blade SKU65302',
 '  #XB-66995 brakepad **',
 '  #XX-47553 battery']

In [15]:

import tiktoken

encoding = tiktoken.encoding_for_model("text-embedding-3-small")

combined_text = " ".join(map(str, dirty_names))

total_tokens = len(encoding.encode(combined_text))
total_cost = total_tokens * (0.02 / 1_000_000)

print(f"Total Unique Items Evaluated: {len(dirty_names):,}")
print(f"Total Tokens: {total_tokens:,}")
print(f"Estimated Cost: ${total_cost:.8f}")

Total Unique Items Evaluated: 1,000
Total Tokens: 7,458
Estimated Cost: $0.00014916


In [16]:
with open(r"ENTER PATH NAME HERE", 'r') as f:
    key_API = f.read().strip()

client = openai.OpenAI(api_key=key_API, base_url='https://openrouter.ai/api/v1')
print("OpenRouter client initialized successfully!")

OpenRouter client initialized successfully!


In [17]:
total_cost = 0

In [18]:
dirty_names = [x for x in dirty_names if isinstance(x, str) and x.strip() != ""]

In [19]:
# embed in groups of 2,048
batch_size = 2048
batched_dirty_names = [dirty_names[i:(i+batch_size)] for i in range(0, len(dirty_names), batch_size)]

dirty_embeddings = []

import tqdm
for batch in tqdm.tqdm(batched_dirty_names):
  e = client.embeddings.create(input = batch, model="openai/text-embedding-3-small")
  dirty_embeddings.extend([i.embedding for i in e.data])

total_cost += e.usage.prompt_tokens * (0.2 / 1000000)

100%|██████████| 1/1 [00:04<00:00,  4.80s/it]


In [20]:
for i, x in enumerate(dirty_names):
    if not isinstance(x, str):
        print(i, repr(x))

In [21]:
print(total_cost)

0.0015024


In [22]:
for i, x in enumerate(map_names):
    if not isinstance(x, str):
        print(i, repr(x))

In [23]:
batch_size = 2048
batched_map_names = [map_names[i:(i+batch_size)] for i in range(0, len(map_names), batch_size)]

map_embeddings = []

import tqdm
for batch in tqdm.tqdm(batched_map_names):
  e = client.embeddings.create(input = batch, model="text-embedding-3-small")
  map_embeddings.extend([i.embedding for i in e.data])

total_cost += e.usage.prompt_tokens * (0.2 / 1000000)

100%|██████████| 1/1 [00:00<00:00,  1.72it/s]


In [24]:
import sklearn.metrics.pairwise as sk_p

distances = sk_p.cosine_distances(map_embeddings, dirty_embeddings)

distances


array([[0.69292595, 0.67898111, 0.71874388, ..., 0.6276042 , 0.66546946,
        0.68142507],
       [0.65525749, 0.68114991, 0.70010905, ..., 0.6234295 , 0.63622277,
        0.69237438],
       [0.58121489, 0.67674467, 0.66107415, ..., 0.61393148, 0.67865728,
        0.6020187 ],
       ...,
       [0.68997982, 0.70198337, 0.7731421 , ..., 0.69273136, 0.74162789,
        0.73406838],
       [0.76068534, 0.39921571, 0.78487215, ..., 0.7268579 , 0.73605682,
        0.76210517],
       [0.30880325, 0.70464266, 0.2977443 , ..., 0.12813488, 0.24380016,
        0.26627226]])

In [25]:
distances.shape

(10, 1000)

In [26]:
dirty_names[6]

'  1511 RAD.'

In [27]:
import numpy as np

closest_names = np.argsort(distances, axis =0)
closest_names

array([[9, 8, 9, ..., 9, 9, 9],
       [2, 4, 4, ..., 4, 4, 2],
       [4, 6, 2, ..., 2, 1, 4],
       ...,
       [6, 3, 5, ..., 3, 8, 7],
       [8, 7, 7, ..., 8, 7, 8],
       [5, 9, 8, ..., 5, 5, 5]], dtype=int64)

In [31]:
dirty_names[270]
#Look at Part 270

'5321 aluminum radiator (discontinued)'

In [32]:
closest_names[:10 , 30]

array([6, 0, 5, 8, 9, 7, 4, 1, 2, 3], dtype=int64)

In [34]:
potential_matches = [map_names[i] for i in closest_names[:10, 270]]
potential_matches
#See how close the embeddings got to finding the correct generic part

['Radiator',
 'Alternator',
 'Headlight Bulb',
 'Brake Pad',
 'Air Filter',
 'Oil Filter',
 'Wiper Blade',
 'Car Battery',
 'Spark Plug',
 'Motor Oil']

In [35]:
#Add a checkpoint t orevert back to
import pickle, os, tqdm

checkpoint_path = 'checkpoint2.pkl'

if os.path.exists(checkpoint_path):
    with open(checkpoint_path, 'rb') as f:
        state = pickle.load(f)
    cleaned_names = state['cleaned_names']
    cleaned_name_ranks = state['cleaned_name_ranks']
    total_cost = state['total_cost']
    start_idx = len(cleaned_names)
    print(f"Resuming from index {start_idx}")
else:
    cleaned_names = []
    cleaned_name_ranks = []
    total_cost = 0
    start_idx = 0

checkpoint_every = 100
#try nano first
for dirty_name_n in tqdm.tqdm(range(start_idx, len(dirty_names)), initial=start_idx, total=len(dirty_names)):
    potential_matches = [map_names[i] for i in closest_names[:5, dirty_name_n]]

    out = client.chat.completions.create(
        model = 'openai/gpt-5.4-nano',
        messages = [{'role':'user', 'content': 'Consider this part name, which may be misspelled or may be the colloquial '
                                            f'name of a part: {dirty_names[dirty_name_n]}'},
                    {'role':'user', 'content': 'Which of the following part names are most likely to be described by this '
                                                "name: " + ', '.join(potential_matches)},
                    {'role':'user', 'content': "Please output the name of one of the parts above *only*, or 'none' if none "
                                                             "of the parts above match."}],
    )
    total_cost += (out.usage.prompt_tokens * (0.4/1000000)) + (out.usage.completion_tokens * (1.6/1000000))
    out_text = out.choices[0].message.content

    if out_text not in potential_matches:
        potential_matches = [map_names[i] for i in closest_names[:10, dirty_name_n]]
#If the model hallucinates a match not in our map names try again 10 times
        tries = 0
        while (out_text not in potential_matches) and (tries < 5):
            out = client.chat.completions.create(
                model = 'openai/gpt-5.4-nano',
                messages = [{'role':'user', 'content': 'Consider this part name, which may be misspelled or may be the colloquial '
                                            f'name of a part: {dirty_names[dirty_name_n]}'},
                    {'role':'user', 'content': 'Which of the following part names are most likely to be described by this '
                                                "name: " + ', '.join(potential_matches)},
                    {'role':'user', 'content': "Please output the name of one of the parts above *only*, or 'none' if none "
                                               "of the parts above match."}]
            )

            total_cost += (out.usage.prompt_tokens * (0.4/1000000)) + (out.usage.completion_tokens * (1.6/1000000))
            out_text = out.choices[0].message.content

            tries += 1
#If the model hallucinates a match not in our map names try again using mini
    if out_text not in potential_matches:
        potential_matches = [map_names[i] for i in closest_names[:10, dirty_name_n]]

        out = client.chat.completions.create(
                model = 'openai/gpt-5.4-mini',
                messages = [{'role':'user', 'content': 'Consider this part name, which may be misspelled or may be the colloquial '
                                            f'name of a part: {dirty_names[dirty_name_n]}'},
                    {'role':'user', 'content': 'Which of the following part names are most likely to be described by this '
                                                "name: " + ', '.join(potential_matches)},
                    {'role':'user', 'content': "Please output the name of one of the parts above *only*, or 'none' if none "
                                               "of the parts above match."}]
        )

        total_cost += (out.usage.prompt_tokens * (0.75/1000000)) + (out.usage.completion_tokens * (10/1000000))
        out_text = out.choices[0].message.content

    if out_text in potential_matches:
        cleaned_names.append(out_text)
        cleaned_name_ranks.append(potential_matches.index(out_text))
    else:
        cleaned_names.append(None)
        cleaned_name_ranks.append(None)

    if (dirty_name_n + 1) % checkpoint_every == 0:
        with open(checkpoint_path, 'wb') as f:
            pickle.dump({
                'cleaned_names': cleaned_names,
                'cleaned_name_ranks': cleaned_name_ranks,
                'total_cost': total_cost
            }, f)

# final save when it finishes
with open(checkpoint_path, 'wb') as f:
    pickle.dump({
        'cleaned_names': cleaned_names,
        'cleaned_name_ranks': cleaned_name_ranks,
        'total_cost': total_cost
    }, f)

100%|██████████| 1000/1000 [13:30<00:00,  1.23it/s]


In [36]:
df_name_cleaner = pd.DataFrame({'Part Name' : dirty_names,
                                'Cleaned Name' : cleaned_names,
                                'Cleaned Rank' : cleaned_name_ranks})

In [37]:
print(len(df_name_cleaner))
df_name_cleaner = df_name_cleaner[df_name_cleaner['Cleaned Name'].notnull()]
print(len(df_name_cleaner))

1000
1000


In [38]:
#Join original part names with their cleaned name to use as a mapping reference table
df_final = pd.merge(df_dirty_names,
                     df_name_cleaner[['Part Name', 'Cleaned Name']],
                     how = 'inner')

df_final.head()

,Part Name,Cleaned Name
0,#MM-18389 Wiper Blade SKU65302,Wiper Blade
1,1434 Engine Air Filter,Air Filter
2,NorthStar cartridge oil filter - universal fit,Oil Filter
3,1711 CABIN AIR FILTER,Air Filter
4,2139 HD Alt.,Alternator


In [39]:
df_final.to_excel('TableOutput.xlsx', index=False)